# 📊 NeuralRetail — Drift Detection
## Amdox Technologies | Data Science & Analytics
---
### What this notebook does:
- Compares current data distribution vs reference (training) data
- Detects if model inputs have drifted over time
- Generates HTML drift report using Evidently AI

In [6]:
# ============================================================
# CELL 2: Imports
# ============================================================
import pandas as pd
import numpy as np
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

✅ Libraries imported!


In [7]:
# ============================================================
# CELL 3: Load data and split into reference vs current
# ============================================================
# Reference = first 70% of data (training period)
# Current   = last 30% of data (production period)
# We check if current data looks different from reference

rfm = pd.read_csv('../data/rfm_features.csv')

# Select numeric features only for drift detection
features = ['Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score']
data = rfm[features].copy()

# Split
split_idx  = int(len(data) * 0.7)
reference  = data.iloc[:split_idx]
current    = data.iloc[split_idx:]

print(f"✅ Data split complete!")
print(f"   Reference (training) : {len(reference):,} customers")
print(f"   Current (production) : {len(current):,} customers")

✅ Data split complete!
   Reference (training) : 3,036 customers
   Current (production) : 1,302 customers


In [8]:
# ============================================================
# CELL 4: Generate Evidently Drift Report
# ============================================================

# Create report with drift + quality metrics
report = Report(metrics=[
    DataDriftPreset(),
    DataQualityPreset()
])

# Run the report
report.run(reference_data=reference, current_data=current)

# Save as HTML
report_path = '../evidently_reports/drift_report.html'
report.save_html(report_path)

print(f"✅ Drift report generated!")
print(f"   Saved to: {report_path}")
print(f"\n📂 Open the file in your browser to view the full report")

✅ Drift report generated!
   Saved to: ../evidently_reports/drift_report.html

📂 Open the file in your browser to view the full report


In [9]:
# ============================================================
# CELL 5: Extract drift summary
# ============================================================

# Get drift results as dictionary
report_dict = report.as_dict()

# Extract drift info per feature
drift_results = report_dict['metrics'][0]['result']
share_drifted = drift_results.get('share_of_drifted_columns', 0)
n_drifted     = drift_results.get('number_of_drifted_columns', 0)

print("=" * 50)
print("📊 DRIFT DETECTION SUMMARY")
print("=" * 50)
print(f"   Total Features  : {len(features)}")
print(f"   Drifted Features: {n_drifted}")
print(f"   Drift Share     : {share_drifted*100:.1f}%")

if share_drifted > 0.5:
    print(f"\n⚠️  WARNING: Significant drift detected!")
    print(f"   Action: Model retraining recommended")
else:
    print(f"\n✅ Data drift is within acceptable range")
    print(f"   Action: No retraining needed")

📊 DRIFT DETECTION SUMMARY
   Total Features  : 7
   Drifted Features: 1
   Drift Share     : 14.3%

✅ Data drift is within acceptable range
   Action: No retraining needed


## ✅ Drift Detection Complete

### What was generated:
- **drift_report.html** — full interactive drift report
- PSI scores per feature showing distribution shift
- Data quality metrics for current vs reference data

### How to use in production:
- Run this notebook weekly
- If drift share > 50% → retrain models
- Share HTML report with stakeholders
